<a href="https://colab.research.google.com/github/estefania-apaza/DocSM-Uribe/blob/main/01_preEHA_GitHub_DocSM_Uribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# preEHA · Ministros de Educación (1980–2026)

El objetivo es preparar `eha_procesada.csv` para el análisis de supervivencia.

El orden sigue el preprocesamiento de la guía del curso: cargar datos, revisar fechas y situación final, crear duración y evento, codificar variables y guardar la base.

## preEHA-01 — Leer los datos sin procesar

In [16]:

import pandas as pd
import numpy as np
from IPython.display import display

# Cargamos la base desde GitHub
url_raw = "https://raw.githubusercontent.com/estefania-apaza/DocSM-Uribe/refs/heads/main/base_PREprocesamiento.csv"
df = pd.read_csv(url_raw)

# Revisamos la base
print("Gestiones:", len(df))
display(df.head(4))

Gestiones: 49


,id_gestion,ministro,fecha_inicio,fecha_fin_observacion,situacion_final,presidente,tipo_designacion,cuestionamiento_congreso,salida_respecto_presidente,vinculo_partido_gobierno,edad,otros_eventos,huelgas
0,1,Luis Felipe Alarco Larrabure,1980-07-28,1981-02-04,Fin de gestión,Fernando Sergio Marcelo Marcos Belaúnde Terry ...,Constitucional,No cuestionado,Antes del presidente,Partido de gobierno,67,Sí,No
1,2,José Benavides Muñoz,1981-02-04,1983-08-15,Fin de gestión,Fernando Sergio Marcelo Marcos Belaúnde Terry ...,Constitucional,No cuestionado,Antes del presidente,Partido de gobierno,54,Sí,No
2,3,Patricio Ricketts Rey de Castro,1983-08-15,1984-04-10,Fin de gestión,Fernando Sergio Marcelo Marcos Belaúnde Terry ...,Constitucional,No cuestionado,Antes del presidente,Partido de gobierno,59,Sí,No
3,4,Valentín Paniagua Corazao,1984-04-10,1984-10-12,Fin de gestión,Fernando Sergio Marcelo Marcos Belaúnde Terry ...,Constitucional,No cuestionado,Antes del presidente,Partido de gobierno,48,No,Sí


## preEHA-02 — Construir duración y evento

In [17]:
# Convertimos las fechas
df["fecha_inicio"] = pd.to_datetime(df["fecha_inicio"])
df["fecha_fin_observacion"] = pd.to_datetime(df["fecha_fin_observacion"])

# Calculamos la duración en meses
df["duracion_meses"] = (
    (df["fecha_fin_observacion"] - df["fecha_inicio"]).dt.days
    / 30.4375
).round(1)

# Todas las gestiones registradas finalizaron
df["caida_evento"] = 1

df[["duracion_meses", "caida_evento"]].head()

,duracion_meses,caida_evento
0,6.3,1
1,30.3,1
2,7.9,1
3,6.1,1
4,9.5,1


In [18]:
# Calculamos cuánto duró cada gestión
df["duracion_dias"] = (
    df["fecha_fin_observacion"] - df["fecha_inicio"]
).dt.days

df["duracion_meses"] = (
    df["duracion_dias"] / 30.4375
).round(1)

# Todas las gestiones registradas finalizaron
df["salida_evento"] = 1

# Revisamos los resultados
display(
    df[["ministro", "duracion_dias", "duracion_meses", "salida_evento"]].head()
)

,ministro,duracion_dias,duracion_meses,salida_evento
0,Luis Felipe Alarco Larrabure,191,6.3,1
1,José Benavides Muñoz,922,30.3,1
2,Patricio Ricketts Rey de Castro,239,7.9,1
3,Valentín Paniagua Corazao,185,6.1,1
4,Andrés Cardó Franco,289,9.5,1


## preEHA-03 — Codificamos las categorías

In [19]:
# Tipo de designación: 1 = constitucional
df["designacion_constitucional"] = (
    df["tipo_designacion"] == "Constitucional"
).astype(int)

# Cuestionamiento del Congreso: 1 = cuestionado
df["congreso_cuestiona"] = (
    df["cuestionamiento_congreso"] == "Cuestionado"
).astype(int)

# Salida respecto al presidente: 1 = sale con el presidente
df["salida_con_presidente"] = (
    df["salida_respecto_presidente"] == "Con el presidente"
).astype(int)

# Vínculo partidario: 1 = pertenece al partido de gobierno
df["vinculo_partidario"] = (
    df["vinculo_partido_gobierno"] == "Partido de gobierno"
).astype(int)

# Otros eventos: 1 = sí
df["otros_eventos_bin"] = (
    df["otros_eventos"] == "Sí"
).astype(int)

# Huelgas: 1 = sí
df["huelgas_bin"] = (
    df["huelgas"] == "Sí"
).astype(int)

# Revisamos las nuevas variables
display(df[[
    "designacion_constitucional",
    "congreso_cuestiona",
    "salida_con_presidente",
    "vinculo_partidario",
    "otros_eventos_bin",
    "huelgas_bin"
]].head())

,designacion_constitucional,congreso_cuestiona,salida_con_presidente,vinculo_partidario,otros_eventos_bin,huelgas_bin
0,1,0,0,1,1,0
1,1,0,0,1,1,0
2,1,0,0,1,1,0
3,1,0,0,1,0,1
4,1,0,1,1,1,0


## preEHA-04 — Guardar la base lista para EHA

In [20]:
# Seleccionamos las variables para el análisis de supervivencia
eha = df[
    [
        "id_gestion",
        "duracion_meses",
        "salida_evento",
        "designacion_constitucional",
        "congreso_cuestiona",
        "salida_con_presidente",
        "vinculo_partidario",
        "edad",
        "otros_eventos_bin",
        "huelgas_bin"
    ]
].copy()

# Guardamos la base procesada
eha.to_csv("eha_procesada.csv", index=False)

# Descargamos el archivo
from google.colab import files
files.download("eha_procesada.csv")

eha.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,id_gestion,duracion_meses,salida_evento,designacion_constitucional,congreso_cuestiona,salida_con_presidente,vinculo_partidario,edad,otros_eventos_bin,huelgas_bin
0,1,6.3,1,1,0,0,1,67,1,0
1,2,30.3,1,1,0,0,1,54,1,0
2,3,7.9,1,1,0,0,1,59,1,0
3,4,6.1,1,1,0,0,1,48,0,1
4,5,9.5,1,1,0,1,1,62,1,0
